# Transportation Planning Simulation

## Objective

This notebook simulates the transportation planning process for disaster response after:

- Resource allocation
- Warehouse allocation
- Shelter allocation

For every disaster event, the notebook determines:

- Which warehouse dispatches relief supplies
- Which shelters receive evacuated people
- Appropriate transport mode
- Estimated travel distance
- Estimated travel time
- Number of rescue vehicles
- Number of supply trucks
- Number of trips required

The transportation plan is designed to imitate realistic disaster-response logistics.

---

## Inputs

- shelter_allocation.csv
- disaster_with_warehouse.csv
- inventory.csv

---

## Output

- transportation_plan.csv
- updated_inventory.csv

This dataset will later be used by the Final Disaster Response Simulation notebook.

In [165]:
# ============================================================
# Imports
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import math

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

In [166]:
# ============================================================
# Helper Function
# ============================================================

def human_number(value):

    value = int(value)

    if value >= 10000000:

        return f"{value:,} ({value/10000000:.2f} Crore)"

    elif value >= 100000:

        return f"{value:,} ({value/100000:.2f} Lakh)"

    elif value >= 1000:

        return f"{value:,} ({value/1000:.2f} Thousand)"

    else:

        return f"{value:,}"

In [167]:
# ============================================================
# Helper Function
# ============================================================

def print_resource_summary(title, value):

    print()

    print(f"{title}")

    print(

        human_number(value)

    )

In [168]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_ROOT = Path.cwd().parents[1]

DATA_DIR = PROJECT_ROOT / "datasets"

OUTPUT_DIR = DATA_DIR

print(PROJECT_ROOT)

C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination


In [169]:
# ============================================================
# Load Simulation Datasets
# ============================================================

events = pd.read_csv(

    DATA_DIR / "shelter_allocation.csv"

)

warehouse = pd.read_csv(

    DATA_DIR / "disaster_with_warehouse.csv"

)

inventory = pd.read_csv(

    DATA_DIR / "inventory.csv"

)

warehouse_columns = [

    "SourceEventID",

    "AssignedWarehouseID",

    "AssignedWarehouse",

    "WarehouseState",

    "DistanceToWarehouseKm"

]

events = events.merge(

    warehouse[warehouse_columns],

    on="SourceEventID",

    how="left"

)

print()

print("Simulation Events :", events.shape)

print("Warehouse Inventory :", inventory.shape)


Simulation Events : (2374, 46)
Warehouse Inventory : (160, 7)


In [170]:
# ============================================================
# Verify Required Columns
# ============================================================

required_event_columns = [

    "SourceEventID",

    "AffectedPopulation",

    "Children",

    "Adults",

    "Elderly",

    "AssignedWarehouse",

    "DistanceToWarehouseKm",

    "AssignedShelterIDs",

    "AverageShelterDistanceKm"

]

missing = [

    c

    for c in required_event_columns

    if c not in events.columns

]

assert len(missing) == 0, missing

required_inventory_columns = [

    "WarehouseID",

    "WarehouseName",

    "Resource",

    "AvailableQuantity"

]

missing_inventory = [

    c

    for c in required_inventory_columns

    if c not in inventory.columns

]

assert len(missing_inventory) == 0, missing_inventory

print()

print("All required columns verified.")


All required columns verified.


In [171]:
# ============================================================
# Transportation Configuration
# ============================================================

TRANSPORT = {

    "Truck": {

        "speed": 55,

        "food_capacity": 4000,

        "medical_capacity": 800,

        "people_capacity": 0,

        "max_distance":600

    },

    "Mini Truck": {

        "speed": 45,

        "food_capacity": 1500,

        "medical_capacity": 300,

        "people_capacity": 0

    },

    "Rescue Bus": {

        "speed": 50,

        "people_capacity": 50

    },

    "Ambulance": {

        "speed": 70,

        "people_capacity": 4

    },

    "Helicopter": {

        "speed": 220,

        "people_capacity": 20

    }

}

print()

print("Transportation Fleet")

for vehicle in TRANSPORT:

    print(vehicle)


Transportation Fleet
Truck
Mini Truck
Rescue Bus
Ambulance
Helicopter


In [172]:
# ============================================================
# Resource Planning Configuration
# ============================================================

DISASTER_DURATION_DAYS = 3

RESOURCE_RULES = {

    "FoodPacketsPerPersonPerDay": 3,

    "WaterLitresPerPersonPerDay": 5,

    "BlanketPerPerson": 1,

    "TentCapacity": 5,

    "MedicalKitChildRate": 0.10,

    "MedicalKitAdultRate": 0.05,

    "MedicalKitElderlyRate": 0.15

}

print()

print("Resource Planning Configuration")

print()

print(f"Disaster Duration : {DISASTER_DURATION_DAYS} Days")

print()

for key, value in RESOURCE_RULES.items():

    print(f"{key:<35}: {value}")


Resource Planning Configuration

Disaster Duration : 3 Days

FoodPacketsPerPersonPerDay         : 3
WaterLitresPerPersonPerDay         : 5
BlanketPerPerson                   : 1
TentCapacity                       : 5
MedicalKitChildRate                : 0.1
MedicalKitAdultRate                : 0.05
MedicalKitElderlyRate              : 0.15


In [173]:
# ============================================================
# Calculate Resource Requirements
# ============================================================

events["FoodPacketsRequired"] = (

    events["AffectedPopulation"]

    *

    RESOURCE_RULES["FoodPacketsPerPersonPerDay"]

    *

    DISASTER_DURATION_DAYS

).round().astype(int)

events["WaterLitresRequired"] = (

    events["AffectedPopulation"]

    *

    RESOURCE_RULES["WaterLitresPerPersonPerDay"]

    *

    DISASTER_DURATION_DAYS

).round().astype(int)

events["BlanketsRequired"] = (

    events["AffectedPopulation"]

    *

    RESOURCE_RULES["BlanketPerPerson"]

).round().astype(int)

events["TentsRequired"] = np.ceil(

    events["AffectedPopulation"]

    /

    RESOURCE_RULES["TentCapacity"]

).astype(int)

events["MedicalKitsRequired"] = (

    events["Children"]

    *

    RESOURCE_RULES["MedicalKitChildRate"]

    +

    events["Adults"]

    *

    RESOURCE_RULES["MedicalKitAdultRate"]

    +

    events["Elderly"]

    *

    RESOURCE_RULES["MedicalKitElderlyRate"]

).apply(np.ceil).astype(int)

print()

print("Resource Demand Calculated")


Resource Demand Calculated


In [174]:
# ============================================================
# Resource Demand Preview
# ============================================================

columns = [

    "SourceEventID",

    "AffectedPopulation",

    "Children",

    "Adults",

    "Elderly",

    "FoodPacketsRequired",

    "MedicalKitsRequired",

    "WaterLitresRequired",

    "BlanketsRequired",

    "TentsRequired"

]

display(

    events[columns].head(10)

)

,SourceEventID,AffectedPopulation,Children,Adults,Elderly,FoodPacketsRequired,MedicalKitsRequired,WaterLitresRequired,BlanketsRequired,TentsRequired
0,18251,182813,49360,109688,23766,1645317,13986,2742195,182813,36563
1,5900,204962,55340,122977,26645,1844658,15680,3074430,204962,40993
2,3158,382680,103324,229608,49748,3444120,29276,5740200,382680,76536
3,13493,1217,329,730,158,10953,94,18255,1217,244
4,4603,433001,116910,259801,56290,3897009,33125,6495015,433001,86601
5,409,518591,140020,311155,67417,4667319,39673,7778865,518591,103719
6,11153,8928146,2410599,5356888,1160659,80353314,683004,133922190,8928146,1785630
7,5396,229919,62078,137951,29889,2069271,17589,3448785,229919,45984
8,18226,108650,29336,65190,14124,977850,8312,1629750,108650,21730
9,17263,5941,1604,3565,772,53469,455,89115,5941,1189


In [175]:
# ============================================================
# Resource Demand Summary
# ============================================================

RESOURCE_NAMES = {

    "FoodPacketsRequired": "Food Packets",

    "MedicalKitsRequired": "Medical Kits",

    "WaterLitresRequired": "Water Bottles",

    "BlanketsRequired": "Blankets",

    "TentsRequired": "Temporary Tents"

}

print()

print("=" * 75)

print("National Resource Requirement")

print("=" * 75)

print()

resources = [

    "FoodPacketsRequired",

    "MedicalKitsRequired",

    "WaterLitresRequired",

    "BlanketsRequired",

    "TentsRequired"

]

for resource in resources:

    total = int(

        events[resource].sum()

    )

    print(

        f"{RESOURCE_NAMES[resource]:<25}: {human_number(total)}"

    )

print()

print(

    f"Total Disaster Events : {len(events):,}"

)

print(

    f"Total Affected Population : {human_number(events['AffectedPopulation'].sum())}"

)


National Resource Requirement

Food Packets             : 1,767,156,921 (176.72 Crore)
Medical Kits             : 15,022,008 (1.50 Crore)
Water Bottles            : 2,945,261,535 (294.53 Crore)
Blankets                 : 196,350,769 (19.64 Crore)
Temporary Tents          : 39,271,152 (3.93 Crore)

Total Disaster Events : 2,374
Total Affected Population : 196,350,769 (19.64 Crore)


In [176]:
# ============================================================
# Create Warehouse Inventory Lookup
# ============================================================

inventory_lookup = {}

for warehouse_name, group in inventory.groupby("WarehouseName"):

    inventory_lookup[warehouse_name] = {

        row["Resource"]: int(row["AvailableQuantity"])

        for _, row in group.iterrows()

    }

print()

print("Warehouses Loaded :", len(inventory_lookup))

print()

print("Example")

first = next(iter(inventory_lookup))

print(first)

print(inventory_lookup[first])


Warehouses Loaded : 16

Example
Ahmedabad Warehouse
{'Food Packets': 2000000, 'Water Bottles': 3000000, 'Medical Kits': 400000, 'Blankets': 600000, 'Temporary Tents': 80000, 'Rescue Teams': 300, 'Ambulances': 300, 'Trucks': 300, 'Boats': 80, 'Helicopters': 25}


In [177]:
# ============================================================
# Inventory Resource Mapping
# ============================================================

RESOURCE_MAPPING = {

    "FoodPacketsRequired": "Food Packets",

    "MedicalKitsRequired": "Medical Kits",

    "WaterLitresRequired": "Water Bottles",

    "BlanketsRequired": "Blankets",

    "TentsRequired": "Temporary Tents"

}

In [178]:
# ============================================================
# Allocate Inventory
# ============================================================

def allocate_inventory(event):

    warehouse = event["AssignedWarehouse"]

    stock = inventory_lookup.get(

        warehouse,

        {}

    )

    allocation = {}

    shortages = {}

    allocated_resources = 0

    partial_resources = 0

    total_resources = len(RESOURCE_MAPPING)

    for demand_column, inventory_resource in RESOURCE_MAPPING.items():

        required = int(event[demand_column])

        available = int(

            stock.get(

                inventory_resource,

                0

            )

        )

        allocated = min(

            required,

            available

        )

        # Deduct inventory from warehouse

        stock[inventory_resource] = (

            available

            -

            allocated

        )

        allocation[demand_column] = allocated

        shortages[demand_column] = (

            required

            -

            allocated

        )

        # Track allocation status

        if allocated > 0:

            allocated_resources += 1

        if allocated < required:

            partial_resources += 1

    # Overall Inventory Status

    if allocated_resources == 0:

        status = "Unavailable"

    elif partial_resources == 0:

        status = "Complete"

    else:

        status = "Partial"

    return pd.Series({

        "FoodAllocated":

            allocation["FoodPacketsRequired"],

        "MedicalAllocated":

            allocation["MedicalKitsRequired"],

        "WaterAllocated":

            allocation["WaterLitresRequired"],

        "BlanketsAllocated":

            allocation["BlanketsRequired"],

        "TentsAllocated":

            allocation["TentsRequired"],

        "FoodShortage":

            shortages["FoodPacketsRequired"],

        "MedicalShortage":

            shortages["MedicalKitsRequired"],

        "WaterShortage":

            shortages["WaterLitresRequired"],

        "BlanketsShortage":

            shortages["BlanketsRequired"],

        "TentsShortage":

            shortages["TentsRequired"],

        "InventoryStatus":

            status

    })

In [179]:
# ============================================================
# Allocate Inventory
# ============================================================

print()

print("Allocating Warehouse Inventory...")

inventory_results = events.apply(

    allocate_inventory,

    axis=1

)

events = pd.concat(

    [

        events,

        inventory_results

    ],

    axis=1

)

print()

print("Inventory Allocation Completed")


Allocating Warehouse Inventory...

Inventory Allocation Completed


In [180]:
# ============================================================
# Inventory Allocation Summary
# ============================================================

print()

print("=" * 75)

print("Inventory Allocation Summary")

print("=" * 75)

print()

print(

    events["InventoryStatus"]

    .value_counts()

)

print_resource_summary(

    "Total Food Allocated",

    events["FoodAllocated"].sum()

)

print_resource_summary(

    "Total Medical Kits Allocated",

    events["MedicalAllocated"].sum()

)

print_resource_summary(

    "Total Water Bottles Allocated",

    events["WaterAllocated"].sum()

)

print_resource_summary(

    "Total Blankets Allocated",

    events["BlanketsAllocated"].sum()

)

print_resource_summary(

    "Total Temporary Tents Allocated",

    events["TentsAllocated"].sum()

)


Inventory Allocation Summary

InventoryStatus
Unavailable    1408
Partial         942
Complete         24
Name: count, dtype: int64

Total Food Allocated
27,200,132 (2.72 Crore)

Total Medical Kits Allocated
3,558,685 (35.59 Lakh)

Total Water Bottles Allocated
41,000,220 (4.10 Crore)

Total Blankets Allocated
7,863,512 (78.64 Lakh)

Total Temporary Tents Allocated
1,066,670 (10.67 Lakh)


In [181]:
print()

print("After Transportation Deduction")

print()

print(inventory_lookup["Ahmedabad Warehouse"])

print()

print(inventory_lookup["Kolkata Warehouse"])


After Transportation Deduction

{'Food Packets': 0, 'Water Bottles': 0, 'Medical Kits': 0, 'Blankets': 0, 'Temporary Tents': 0, 'Rescue Teams': 300, 'Ambulances': 300, 'Trucks': 300, 'Boats': 80, 'Helicopters': 25}

{'Food Packets': 0, 'Water Bottles': 0, 'Medical Kits': 0, 'Blankets': 0, 'Temporary Tents': 0, 'Rescue Teams': 300, 'Ambulances': 300, 'Trucks': 300, 'Boats': 80, 'Helicopters': 25}


In [182]:
# ============================================================
# National Resource Shortage
# ============================================================

print()

print("=" * 75)

print("National Resource Shortage")

print("=" * 75)

print()

print_resource_summary(

    "Food Shortage",

    events["FoodShortage"].sum()

)

print_resource_summary(

    "Medical Kits Shortage",

    events["MedicalShortage"].sum()

)

print_resource_summary(

    "Water Bottles Shortage",

    events["WaterShortage"].sum()

)

print_resource_summary(

    "Blankets Shortage",

    events["BlanketsShortage"].sum()

)

print_resource_summary(

    "Temporary Tents Shortage",

    events["TentsShortage"].sum()

)

print()

print("=" * 75)

print("Warehouse Utilization")

print("=" * 75)

print()

utilization = (

    events["InventoryStatus"]

    .value_counts(normalize=True)

    * 100

).round(2)

print(utilization)


National Resource Shortage


Food Shortage
1,739,956,789 (174.00 Crore)

Medical Kits Shortage
11,463,323 (1.15 Crore)

Water Bottles Shortage
2,904,261,315 (290.43 Crore)

Blankets Shortage
188,487,257 (18.85 Crore)

Temporary Tents Shortage
38,204,482 (3.82 Crore)

Warehouse Utilization

InventoryStatus
Unavailable    59.31
Partial        39.68
Complete        1.01
Name: proportion, dtype: float64


In [183]:
display(

    events[

        [

            "SourceEventID",

            "AssignedWarehouse",

            "InventoryStatus",

            "FoodAllocated",

            "MedicalAllocated",

            "WaterAllocated",

            "BlanketsAllocated",

            "TentsAllocated",

            "FoodShortage",

            "MedicalShortage"

        ]

    ]

    .head(10)

)

,SourceEventID,AssignedWarehouse,InventoryStatus,FoodAllocated,MedicalAllocated,WaterAllocated,BlanketsAllocated,TentsAllocated,FoodShortage,MedicalShortage
0,18251,Ahmedabad Warehouse,Complete,1645317,13986,2742195,182813,36563,0,0
1,5900,Guwahati Warehouse,Partial,1844658,15680,3000000,204962,40993,0,0
2,3158,Guwahati Warehouse,Partial,155342,29276,0,382680,39007,3288778,0
3,13493,Srinagar Warehouse,Complete,10953,94,18255,1217,244,0,0
4,4603,Guwahati Warehouse,Partial,0,33125,0,12358,0,3897009,0
5,409,Guwahati Warehouse,Partial,0,39673,0,0,0,4667319,0
6,11153,Delhi Central Warehouse,Partial,2000000,400000,3000000,600000,80000,78353314,283004
7,5396,Guwahati Warehouse,Partial,0,17589,0,0,0,2069271,0
8,18226,Ahmedabad Warehouse,Partial,354683,8312,257805,108650,21730,623167,0
9,17263,Port Blair Warehouse,Complete,53469,455,89115,5941,1189,0,0


In [184]:
# ============================================================
# Vehicle Requirement Calculation
# ============================================================

RESOURCE_WEIGHT = {

    "FoodPacket": 0.5,

    "WaterBottle": 1.0,

    "MedicalKit": 0.3,

    "Blanket": 2.5,

    "Tent": 20.0

}

TRUCK_PAYLOAD = 8000

events["TotalSupplyWeightKg"] = (

    events["FoodAllocated"]

    * RESOURCE_WEIGHT["FoodPacket"]

    +

    events["WaterAllocated"]

    * RESOURCE_WEIGHT["WaterBottle"]

    +

    events["MedicalAllocated"]

    * RESOURCE_WEIGHT["MedicalKit"]

    +

    events["BlanketsAllocated"]

    * RESOURCE_WEIGHT["Blanket"]

    +

    events["TentsAllocated"]

    * RESOURCE_WEIGHT["Tent"]

)

events["TotalSupplyTrucks"] = np.ceil(

    events["TotalSupplyWeightKg"]

    /

    TRUCK_PAYLOAD

).astype(int)

In [185]:
# ============================================================
# Transportation Rules
# ============================================================

TRANSPORT_RULES = {

    "TruckFoodCapacity": 2000,

    "TruckBlanketCapacity": 1000,

    "TruckTentCapacity": 80,

    "CriticalInjuryRate": 0.05,

    "AmbulanceCapacity": 4,

    "RescueBusCapacity": 50,

    "RescueTeamCoverage": 250,

    "HelicopterCapacity": 400,

    "BoatCoverage": 5000

}

In [186]:
# ============================================================
# Vehicle Requirement Calculation
# ============================================================

events["FoodTrucksRequired"] = np.ceil(

    events["FoodAllocated"]

    /

    TRANSPORT_RULES["TruckFoodCapacity"]

).astype(int)

events["BlanketTrucksRequired"] = np.ceil(

    events["BlanketsAllocated"]

    /

    TRANSPORT_RULES["TruckBlanketCapacity"]

).astype(int)

events["TentTrucksRequired"] = np.ceil(

    events["TentsAllocated"]

    /

    TRANSPORT_RULES["TruckTentCapacity"]

).astype(int)

events["TotalSupplyTrucks"] = (

    events["FoodTrucksRequired"]

    +

    events["BlanketTrucksRequired"]

    +

    events["TentTrucksRequired"]

)

In [187]:
# ============================================================
# Rescue Transportation
# ============================================================

events["CriticalPatients"] = np.ceil(

    events["AffectedPopulation"]

    *

    TRANSPORT_RULES["CriticalInjuryRate"]

).astype(int)

events["AmbulancesRequired"] = np.ceil(

    events["CriticalPatients"]

    /

    TRANSPORT_RULES["AmbulanceCapacity"]

).astype(int)

events["RemainingEvacuees"] = (

    events["AffectedPopulation"]

    -

    events["CriticalPatients"]

)

events["RescueBusesRequired"] = np.ceil(

    events["RemainingEvacuees"]

    /

    TRANSPORT_RULES["RescueBusCapacity"]

).astype(int)

events["RescueTeamsRequired"] = np.ceil(

    events["AffectedPopulation"]

    /

    TRANSPORT_RULES["RescueTeamCoverage"]

).astype(int)

events["HelicopterSorties"] = np.where(

    (

        events["Severity"] == "High"

    )

    |

    (

        events["DistanceToWarehouseKm"] > 300

    ),

    np.ceil(

        events["CriticalPatients"]

        /

        TRANSPORT_RULES["HelicopterCapacity"]

    ),

    0

).astype(int)

# ============================================================
# Transportation Resource Availability
# ============================================================

TRANSPORT_AVAILABLE = {

    "RescueTeams": 300,

    "Ambulances": 300,

    "Boats": 80,

    "Helicopters": 25

}

events["RescueTeamsAvailable"] = TRANSPORT_AVAILABLE["RescueTeams"]

events["AmbulancesAvailable"] = TRANSPORT_AVAILABLE["Ambulances"]

events["BoatsAvailable"] = TRANSPORT_AVAILABLE["Boats"]

events["HelicoptersAvailable"] = TRANSPORT_AVAILABLE["Helicopters"]

In [188]:
print(inventory.head(20))

   WarehouseID         WarehouseName              State  Latitude  Longitude  \
0        WH001    Srinagar Warehouse  Jammu and Kashmir   34.0837    74.7973   
1        WH001    Srinagar Warehouse  Jammu and Kashmir   34.0837    74.7973   
2        WH001    Srinagar Warehouse  Jammu and Kashmir   34.0837    74.7973   
3        WH001    Srinagar Warehouse  Jammu and Kashmir   34.0837    74.7973   
4        WH001    Srinagar Warehouse  Jammu and Kashmir   34.0837    74.7973   
5        WH001    Srinagar Warehouse  Jammu and Kashmir   34.0837    74.7973   
6        WH001    Srinagar Warehouse  Jammu and Kashmir   34.0837    74.7973   
7        WH001    Srinagar Warehouse  Jammu and Kashmir   34.0837    74.7973   
8        WH001    Srinagar Warehouse  Jammu and Kashmir   34.0837    74.7973   
9        WH001    Srinagar Warehouse  Jammu and Kashmir   34.0837    74.7973   
10       WH002  Chandigarh Warehouse         Chandigarh   30.7333    76.7794   
11       WH002  Chandigarh Warehouse    

In [189]:
# ============================================================
# Transportation Resource Summary
# ============================================================

print()

print("=" * 75)

print("Transportation Resource Summary")

print("=" * 75)

print()

print_resource_summary(

    "Supply Trucks Required",

    events["TotalSupplyTrucks"].sum()

)

print_resource_summary(

    "Ambulances Required",

    events["AmbulancesRequired"].sum()

)

print_resource_summary(

    "Rescue Buses Required",

    events["RescueBusesRequired"].sum()

)

print_resource_summary(

    "Rescue Teams Required",

    events["RescueTeamsRequired"].sum()

)

print_resource_summary(

    "Helicopter Sorties",

    events["HelicopterSorties"].sum()

)

print_resource_summary(

    "Critical Patients",

    events["CriticalPatients"].sum()

)

print_resource_summary(

    "People Requiring Evacuation",

    events["RemainingEvacuees"].sum()

)


Transportation Resource Summary


Supply Trucks Required
34,878 (34.88 Thousand)

Ambulances Required
2,455,655 (24.56 Lakh)

Rescue Buses Required
3,731,843 (37.32 Lakh)

Rescue Teams Required
786,714 (7.87 Lakh)

Helicopter Sorties
2,091 (2.09 Thousand)

Critical Patients
9,818,732 (98.19 Lakh)

People Requiring Evacuation
186,532,037 (18.65 Crore)


In [190]:
# ============================================================
# Estimate Travel Time
# ============================================================

def estimate_travel_time(distance, vehicle):

    speed = TRANSPORT[vehicle]["speed"]

    return round(

        distance / speed,

        2

    )

In [191]:
# ============================================================
# Travel Time Estimation
# ============================================================

events["SupplyTravelTimeHours"] = events.apply(

    lambda row:

    estimate_travel_time(

        row["DistanceToWarehouseKm"],

        "Truck"

    ),

    axis=1

)

events["AmbulanceTravelTimeHours"] = events.apply(

    lambda row:

    estimate_travel_time(

        row["AverageShelterDistanceKm"],

        "Ambulance"

    ),

    axis=1

)

events["HelicopterTravelTimeHours"] = events.apply(

    lambda row:

    estimate_travel_time(

        row["DistanceToWarehouseKm"],

        "Helicopter"

    ),

    axis=1

)

In [192]:
print()

print("=" * 75)

print("Travel Time Summary")

print("=" * 75)

print()

print(

    f"Average Supply Transport Time      : {events['SupplyTravelTimeHours'].mean():.2f} hours"

)

print(

    f"Average Ambulance Travel Time      : {events['AmbulanceTravelTimeHours'].mean():.2f} hours"

)

print(

    f"Average Helicopter Flight Time     : {events['HelicopterTravelTimeHours'].mean():.2f} hours"

)

print()

print(

    f"Maximum Supply Transport Time      : {events['SupplyTravelTimeHours'].max():.2f} hours"

)

print(

    f"Maximum Ambulance Travel Time      : {events['AmbulanceTravelTimeHours'].max():.2f} hours"

)


Travel Time Summary

Average Supply Transport Time      : 4.25 hours
Average Ambulance Travel Time      : 2.66 hours
Average Helicopter Flight Time     : 1.06 hours

Maximum Supply Transport Time      : 11.06 hours
Maximum Ambulance Travel Time      : 21.12 hours


In [193]:
# ============================================================
# Mission Status
# ============================================================

conditions = [

    (

        events["InventoryStatus"] == "Complete"

    )

    &

    (

        events["AllocationStatus"] == "Fully Allocated"

    ),

    (

        events["InventoryStatus"] == "Partial"

    )

    |

    (

        events["AllocationStatus"] == "Partially Allocated"

    )

]

choices = [

    "Mission Ready",

    "Limited Response"

]

events["MissionStatus"] = np.select(

    conditions,

    choices,

    default="Critical Shortage"

)

In [194]:
print()

print("=" * 75)

print("Mission Status Summary")

print("=" * 75)

print()

mission = (

    events["MissionStatus"]

    .value_counts()

)

print(mission)

print()

print(

    (

        mission

        /

        len(events)

        *100

    ).round(2)

)


Mission Status Summary

MissionStatus
Critical Shortage    1342
Limited Response     1012
Mission Ready          20
Name: count, dtype: int64

MissionStatus
Critical Shortage    56.53
Limited Response     42.63
Mission Ready         0.84
Name: count, dtype: float64


In [195]:
# ============================================================
# Update Transportation Resource Inventory
# ============================================================

TRANSPORT_MAPPING = {

    "Trucks": "TotalSupplyTrucks",

    "Ambulances": "AmbulancesRequired",

    "Rescue Teams": "RescueTeamsRequired",

    "Boats": "BoatsRequired",

    "Helicopters": "HelicopterSorties"

}

for _, event in events.iterrows():

    warehouse = event["AssignedWarehouse"]

    if warehouse not in inventory_lookup:

        continue

    stock = inventory_lookup[warehouse]

    for resource, demand_column in TRANSPORT_MAPPING.items():

        if demand_column not in events.columns:

            continue

        required = int(event[demand_column])

        available = int(

            stock.get(resource, 0)

        )

        deployed = min(

            required,

            available

        )

        stock[resource] = (

            available

            -

            deployed

        )


# ============================================================
# Create Updated Warehouse Inventory
# ============================================================

updated_inventory = []

for warehouse_name, resources in inventory_lookup.items():

    for resource, quantity in resources.items():

        updated_inventory.append(

            {

                "WarehouseName": warehouse_name,

                "Resource": resource,

                "RemainingQuantity": quantity

            }

        )

updated_inventory = pd.DataFrame(

    updated_inventory

)

print()

print("Transportation Inventory Updated")

display(

    updated_inventory.head()

)


Transportation Inventory Updated


,WarehouseName,Resource,RemainingQuantity
0,Ahmedabad Warehouse,Food Packets,0
1,Ahmedabad Warehouse,Water Bottles,0
2,Ahmedabad Warehouse,Medical Kits,0
3,Ahmedabad Warehouse,Blankets,0
4,Ahmedabad Warehouse,Temporary Tents,0


In [196]:
# ============================================================
# Save Transportation Plan
# ============================================================

events.to_csv(

    DATA_DIR / "transportation_plan.csv",

    index=False

)

updated_inventory.to_csv(

    DATA_DIR / "updated_inventory.csv",

    index=False

)

print()

print("Transportation Plan Saved")

print(

    DATA_DIR / "transportation_plan.csv"

)

print()

print("Updated Inventory Saved")

print(

    DATA_DIR / "updated_inventory.csv"

)


Transportation Plan Saved
C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination\datasets\transportation_plan.csv

Updated Inventory Saved
C:\Users\aashi\PycharmProjects\AI-Based Disaster Response Management System for Resource Allocation and Relief Coordination\datasets\updated_inventory.csv
